In [1]:


import numpy as np


class LogisticRegression:
    def __init__(self, learning_rate=0.1, n_iters=1000):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.w = None
        self.b = None
        self.loss_history = []

    @staticmethod
    def sigmoid(z):
        # Clip z to avoid overflow in exp() for very negative/positive values
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    @staticmethod
    def binary_cross_entropy(y_true, y_pred):
        # Clip predictions so log(0) never happens
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )

    def fit(self, X, y):
        """
        X: shape (n_samples, n_features)
        y: shape (n_samples,), values 0 or 1
        """
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for i in range(self.n_iters):
            # Forward pass
            z = X @ self.w + self.b
            p = self.sigmoid(z)

            # Loss (just for tracking / plotting convergence)
            loss = self.binary_cross_entropy(y, p)
            self.loss_history.append(loss)

            # Gradients (derived from BCE + sigmoid combo)
            dw = (1 / n_samples) * (X.T @ (p - y))
            db = (1 / n_samples) * np.sum(p - y)

            # Update
            self.w -= self.lr * dw
            self.b -= self.lr * db

            if i % 100 == 0:
                print(f"Iteration {i:4d} | loss = {loss:.4f}")

        return self

    def predict_proba(self, X):
        z = X @ self.w + self.b
        return self.sigmoid(z)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


def confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn


def evaluate(y_true, y_pred):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    print("\nConfusion Matrix:")
    print(f"                Predicted 1   Predicted 0")
    print(f"  Actually 1:   TP={tp:<10}  FN={fn}")
    print(f"  Actually 0:   FP={fp:<10}  TN={tn}")
    print(f"\nAccuracy:  {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


if __name__ == "__main__":
    
    # Toy example: predict pass/fail from hours studied + practice tests
    
    np.random.seed(42)

    # Feature 1: hours studied, Feature 2: practice tests taken
    X = np.array([
        [1, 0], [2, 0], [2, 1], [3, 1], [3, 2],
        [4, 1], [5, 2], [5, 3], [6, 3], [6, 4],
        [7, 3], [7, 4], [8, 4], [9, 5], [10, 5],
    ], dtype=float)

    y = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=float)

    # Normalize features (helps gradient descent converge faster/stably)
    X_mean, X_std = X.mean(axis=0), X.std(axis=0)
    X_norm = (X - X_mean) / X_std

    model = LogisticRegression(learning_rate=0.5, n_iters=1000)
    model.fit(X_norm, y)

    print(f"\nLearned weights: {model.w}")
    print(f"Learned bias:    {model.b:.4f}")

    y_pred = model.predict(X_norm)
    print(f"\nPredictions:  {y_pred}")
    print(f"Actual:       {y.astype(int)}")

    evaluate(y, y_pred)

    # Try a new student: 4 hours studied, 2 practice tests
    new_student = np.array([[4, 2]], dtype=float)
    new_student_norm = (new_student - X_mean) / X_std
    prob = model.predict_proba(new_student_norm)[0]
    print(f"\nNew student (4 hrs, 2 tests) -> P(pass) = {prob:.3f} "
          f"-> predicted: {'PASS' if prob >= 0.5 else 'FAIL'}")

Iteration    0 | loss = 0.6931
Iteration  100 | loss = 0.0752
Iteration  200 | loss = 0.0521
Iteration  300 | loss = 0.0412
Iteration  400 | loss = 0.0344
Iteration  500 | loss = 0.0297
Iteration  600 | loss = 0.0262
Iteration  700 | loss = 0.0235
Iteration  800 | loss = 0.0213
Iteration  900 | loss = 0.0195

Learned weights: [6.08244693 4.16714677]
Learned bias:    3.6972

Predictions:  [0 0 0 0 0 0 1 1 1 1 1 1 1 1 1]
Actual:       [0 0 0 0 0 0 1 1 1 1 1 1 1 1 1]

Confusion Matrix:
                Predicted 1   Predicted 0
  Actually 1:   TP=9           FN=0
  Actually 0:   FP=0           TN=6

Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1 Score:  1.000

New student (4 hrs, 2 tests) -> P(pass) = 0.387 -> predicted: FAIL
